# Map 与 Set

学习目标：能按数据关系选择键值映射、去重集合或弱集合，正确处理对象身份和集合运算。

前置知识：对象属性、数组、函数回调、for...of 与对象引用。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 文件使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/11-map-and-set/。

1. [map-basics.mjs](scripts/11-map-and-set/map-basics.mjs)：Map 增删查与缺失值。
2. [identity-and-order.mjs](scripts/11-map-and-set/identity-and-order.mjs)：对象键、SameValueZero 与遍历。
3. [set-basics.mjs](scripts/11-map-and-set/set-basics.mjs)：去重、集合成员与遍历。
4. [set-operations.mjs](scripts/11-map-and-set/set-operations.mjs)：七种集合运算与结果顺序。
5. [weak-collections.mjs](scripts/11-map-and-set/weak-collections.mjs)：弱映射与弱成员标记。
6. [registered-symbol-error.mjs](scripts/11-map-and-set/registered-symbol-error.mjs)：注册符号不能作为弱键的独立反例。
7. [choose-container.mjs](scripts/11-map-and-set/choose-container.mjs)：用编号合并记录。

## 1 用 Map 保存键值关系

把课程名映射到学习时长时，Map 明确表达“一个键对应一个值”。本例将由二元素数组组成的可迭代对象传给构造函数；每个内部数组依次放键和值。set 写入或覆盖并返回当前 Map，get 读取，has 判断键是否存在，size 是条目数。

get 找不到键时返回 undefined，但已存在的键也可以保存 undefined，因此判断存在性要用 has。delete 返回是否删掉了条目；clear 清空并返回 undefined。给 Map 写普通对象属性不会写入映射条目。

[map-basics.mjs](scripts/11-map-and-set/map-basics.mjs)：

```javascript
const hours = new Map([["语法", 2], ["集合", 3]]);
console.log(hours.set("语法", 4) === hours, hours.size, hours.get("语法"));
hours.set("待安排", undefined);
console.log(hours.get("待安排"), hours.has("待安排"), hours.has("不存在"));
hours.note = "普通属性";
console.log(hours.has("note"), hours.note);
console.log(hours.delete("集合"), hours.delete("集合"));
console.log(hours.clear(), hours.size);

// 按本例输入运行，输出依次为：
// true 2 4
// undefined true false
// false 普通属性
// true false
// undefined 0
```

Step 1：运行本节示例。

```bash
node scripts/11-map-and-set/map-basics.mjs
```

## 2 键的相等规则和插入顺序

Map 键和 Set 成员采用 SameValueZero：NaN 与 NaN 视为相等，+0 与 -0 视为相等，其余与严格相等的相关规则一致；不同类型不自动转换。对象比较身份，不比较属性内容。用对象作为键后修改对象属性，仍然是同一个键。

Map 按键首次插入顺序遍历。覆盖已有值不移动位置；删除后再加入会进入末尾。keys、values、entries 分别产生键、值、键值对；默认 for...of 使用 entries。forEach 的回调参数依次是值、键、当前 Map，这与 entries 的二元素顺序不同。遍历不是快照，途中加入的新条目可能被访问，因此本例先修改、后遍历。

[identity-and-order.mjs](scripts/11-map-and-set/identity-and-order.mjs)：

```javascript
const learner = { name: "林" };
const visits = new Map([[learner, 1], [NaN, "首次"], [-0, "零"]]);
learner.name = "林同学";
visits.set(NaN, "更新");
visits.set(+0, "同一个零");
console.log(visits.size, visits.get(learner), visits.has({ name: "林同学" }));
console.log(visits.get(NaN), visits.get(-0), visits.has("0"));
const order = new Map([["a", 1], ["b", 2]]);
order.set("a", 10);
console.log([...order.keys()].join(","));
order.delete("a");
order.set("a", 20);
console.log([...order.values()].join(","));
const pairs = [];
for (const [key, value] of order) pairs.push(key + ":" + value);
console.log(pairs.join(","));
order.forEach((value, key, map) => console.log(key, value, map === order));

// 按本例输入运行，输出依次为：
// 3 1 false
// 更新 同一个零 false
// a,b
// 2,20
// b:2,a:20
// b 2 true
// a 20 true
```

Step 1：运行本节示例。

```bash
node scripts/11-map-and-set/identity-and-order.mjs
```

## 3 用 Set 去重并查询成员

Set 只保存成员，重复 add 不增加 size，add 返回当前集合。has、delete、clear 分别判断、删除、清空；delete 返回布尔值，clear 返回 undefined。Set 按插入顺序遍历，默认迭代器与 values 相同；keys 也是 values 的别名，entries 产生“成员、成员”对，以便与 Map 接口一致。

把数组交给 Set 再展开，可以按首次出现顺序去重。对象内容相同不意味着会去重；如果业务按编号识别记录，应先选择编号作 Map 键，而不是依赖对象内容相等。

[set-basics.mjs](scripts/11-map-and-set/set-basics.mjs)：

```javascript
const topics = new Set(["函数", "集合", "函数"]);
console.log(topics.add("模块") === topics, topics.size, topics.has("集合"));
console.log([...topics].join(","));
console.log(JSON.stringify([...topics.entries()]));
const seen = [];
topics.forEach((value, key) => seen.push(value === key));
console.log(seen.every(Boolean));
console.log(topics.delete("模块"), topics.delete("模块"));
const same = { id: 1 };
console.log(new Set([same, same, { id: 1 }]).size);
console.log(topics.clear(), topics.size);

// 按本例输入运行，输出依次为：
// true 3 true
// 函数,集合,模块
// [["函数","函数"],["集合","集合"],["模块","模块"]]
// true
// true false
// 2
// undefined 0
```

Step 1：运行本节示例。

```bash
node scripts/11-map-and-set/set-basics.mjs
```

## 4 集合运算

以下方法已纳入 ECMAScript 2025；本章固定的 Node.js 支持这些方法。left 和 right 在本例中分别表示已学主题和计划主题。生成集合的方法返回新 Set，不改变输入集合。

| 方法原文名 | 中文名称／含义 | 返回值 |
| --- | --- | --- |
| Set.prototype.union | 并集，任一集合中存在 | 新 Set |
| Set.prototype.intersection | 交集，两边都存在 | 新 Set |
| Set.prototype.difference | 差集，仅保留当前集合独有成员 | 新 Set |
| Set.prototype.symmetricDifference | 对称差，两边仅出现一次的成员 | 新 Set |
| Set.prototype.isSubsetOf | 是否为子集 | boolean |
| Set.prototype.isSupersetOf | 是否为超集 | boolean |
| Set.prototype.isDisjointFrom | 是否没有公共成员 | boolean |

参数采用集合式接口：需要 size、has 和 keys，并非任意可迭代对象都可用；数组不能直接充当参数，Map 可按键参加集合运算。接收方法调用的 this 必须有真正的 Set 内部数据。交集为减少查询次数，会遍历较小的一方；不能笼统地说结果总按左侧顺序排列。

[set-operations.mjs](scripts/11-map-and-set/set-operations.mjs)：

```javascript
const left = new Set(["函数", "集合"]);
const right = new Set(["集合", "模块"]);
console.log([...left.union(right)].join(","));
console.log([...left.intersection(right)].join(","));
console.log([...left.difference(right)].join(","));
console.log([...left.symmetricDifference(right)].join(","));
console.log(left.isSubsetOf(left.union(right)), left.isSupersetOf(new Set(["函数"])));
console.log(left.isDisjointFrom(new Set(["日期"])), left.size, right.size);
console.log([...new Set([1, 2, 3]).intersection(new Set([2, 1]))].join(","));
console.log([...new Set([1, 2]).intersection(new Map([[2, "记录"]]))].join(","));

// 按本例输入运行，输出依次为：
// 函数,集合,模块
// 集合
// 函数
// 函数,模块
// true true
// true 2 2
// 2,1
// 2
```

Step 1：运行本节示例。

```bash
node scripts/11-map-and-set/set-operations.mjs
```

## 5 弱集合保存附属状态

WeakMap 适合把附属信息关联到对象而不通过这一关联单独保活键；WeakSet 适合标记对象是否被处理过。WeakMap 关联的值仍可是任意类型。弱集合可用的键或成员是对象，以及没有在全局符号注册表注册的 Symbol；Symbol.for 产生的注册符号不能使用，字符串、数字、null 也不能使用。Symbol() 创建的符号满足条件。

弱集合只能拿已知键或成员查询，没有 size、keys、values、entries 和遍历接口，也没有 clear；不能观察“垃圾回收后还剩多少”。回收是否发生及发生时间不能作为业务条件。WeakMap 提供 set/get/has/delete，WeakSet 提供 add/has/delete。需要明确删除状态时主动 delete，不等待回收。

[weak-collections.mjs](scripts/11-map-and-set/weak-collections.mjs)：

```javascript
const item = { title: "笔记" };
const metadata = new WeakMap();
metadata.set(item, { reviewed: true });
const processed = new WeakSet([item]);
console.log(metadata.get(item).reviewed, processed.has(item));
const localToken = Symbol("局部令牌");
metadata.set(localToken, 5);
processed.add(localToken);
console.log(metadata.get(localToken), processed.has(localToken));
console.log(metadata.delete(item), processed.delete(item));
console.log(metadata.has(item), processed.has(item));
console.log(typeof metadata[Symbol.iterator], typeof processed.size);

// 按本例输入运行，输出依次为：
// true true
// 5 true
// true true
// false false
// undefined undefined
```

Step 1：运行本节示例。

```bash
node scripts/11-map-and-set/weak-collections.mjs
```

[registered-symbol-error.mjs](scripts/11-map-and-set/registered-symbol-error.mjs)：

```javascript
const metadata = new WeakMap();
metadata.set(Symbol.for("shared"), 1);

// 独立运行：退出状态为 1；诊断包含 TypeError；Invalid value used as weak map key。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/11-map-and-set/registered-symbol-error.mjs
```

## 6 按数据关系选择容器

Object 适合有固定字段的记录，属性键为字符串或 Symbol；Array 适合需要索引、重复元素与位置关系的序列。Map 适合任意类型键到值的映射，Set 适合成员关系与去重。这些是本例的数据建模选择，不是性能排名。

例如同一编号的最新记录覆盖旧记录，可以用 Map；每条记录内部的标题与分钟数仍放 Object，展示顺序转换成 Array。不要把 Map 当 JSON 对象：直接 JSON.stringify(Map) 不会自动序列化条目，JSON 的转换边界在“JSON 与数据转换”章展开。

[choose-container.mjs](scripts/11-map-and-set/choose-container.mjs)：

```javascript
const records = [{ id: "a", minutes: 10 }, { id: "b", minutes: 20 }, { id: "a", minutes: 30 }];
const byId = new Map();
for (const record of records) byId.set(record.id, record);
const latest = [...byId.values()];
console.log(latest.map(record => record.id + ":" + record.minutes).join(","));
console.log(new Set(records.map(record => record.id)).size);

// 按本例输入运行，输出依次为：
// a:30,b:20
// 2
```

Step 1：运行本节示例。

```bash
node scripts/11-map-and-set/choose-container.mjs
```

## 本章小结

- Map 表示键到值的关联，Set 表示成员关系；相等规则不会比较对象内容。
- 插入、覆盖和删除后重插的顺序不同；集合运算返回新结果。
- 弱集合只能查询已知对象或非注册符号，不能据其推断回收时机。

## 练习

1. 为两个对象各记录访问次数，再给第一个对象增加属性。可核对标准：Map 仍有两个键，修改后的第一个对象仍能取回原次数。
2. 对集合 {1, 2, 3} 与 {3, 4} 计算交、并、差集。可核对标准：分别含 {3}、{1, 2, 3, 4}、{1, 2}，两个输入不变。
3. 将 registered-symbol-error.mjs 的 Symbol.for 改为单独保存的 Symbol()，写入后用同一个符号读取。可核对标准：正常退出、读到 1；解释为什么重新调用 Symbol() 不能取回值。

### 提示

1. 保留同一对象变量，再修改属性。
2. 分清结果的新集合与两个输入集合。
3. 创建一次符号，并重复使用该变量。


### 参考解析

1. Map 根据对象身份查找；增加属性不会改变身份，两个不同对象仍占两个键。
2. left.intersection(right)、left.union(right)、left.difference(right) 分别得到 3、1,2,3,4、1,2；输入 size 仍为 3 和 2。
3. `const key = Symbol("shared"); metadata.set(key, 1); metadata.get(key)` 得到 1。再次 Symbol("shared") 创建的是另一枚符号，即使描述相同也不是原键。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TC39 官方 ECMAScript 2025 分页版 | [§24.1 Map 的读写、迭代和 SameValueZero](https://tc39.es/ecma262/2025/multipage/keyed-collections.html#sec-map-objects)；[§24.2 Set 及集合运算](https://tc39.es/ecma262/2025/multipage/keyed-collections.html#sec-set-objects)；[§24.3–24.4 弱集合](https://tc39.es/ecma262/2025/multipage/keyed-collections.html#sec-weakmap-objects)；[§9.13 CanBeHeldWeakly](https://tc39.es/ecma262/2025/multipage/executable-code-and-execution-contexts.html#sec-canbeheldweakly)；[§7.2.10 相等规则](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-samevaluezero)。 |
